# Pharma Revenue Forecasting — Training Notebook

**Goal:** Train a two-stage LightGBM pipeline to forecast monthly revenue per product for May 2026 → April 2027.

**Architecture:**
- Stage 1: LightGBM binary classifier — predicts P(revenue > 0)
- Stage 2: LightGBM regressor — predicts log1p(revenue) on non-zero rows only
- Final pred: `stage1_prob × expm1(stage2_pred)` (or 0 if prob < threshold)


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from pathlib import Path

import lightgbm as lgb
from sklearn.metrics import roc_auc_score, mean_squared_error

pd.set_option('display.float_format', '{:,.2f}'.format)
np.random.seed(42)

DATA_PATH = 'monthly_panel.csv'
MODEL_DIR = Path('models')
MODEL_DIR.mkdir(exist_ok=True)

print('Libraries loaded ✓')

---
## 1. Load Data & Add revenue_lag12

In [ ]:
df = pd.read_csv(DATA_PATH)
df['month_dt'] = pd.to_datetime(df['month_dt'])
df = df.sort_values(['product_id', 'month_dt']).reset_index(drop=True)

# Add revenue_lag12 — same month last year (the #1 predictor)
df['revenue_lag12'] = df.groupby('product_id')['revenue'].shift(12)

print(f'Shape: {df.shape}')
print(f'Date range: {df.month_dt.min().date()} → {df.month_dt.max().date()}')
print(f'Products: {df.product_id.nunique()}')
print(f'revenue_lag12 non-null: {df.revenue_lag12.notna().sum()} / {len(df)}')

---
## 2. Feature Engineering — Lag Context Features

Create lag1 versions of context variables that would otherwise leak future information.

In [ ]:
lag_context_cols = [
    'n_active_clients', 'n_invoices', 'n_regions',
    'export_rev_share', 'hospital_rev_share', 'commission_rate_w', 'unit_price_w'
]

for col in lag_context_cols:
    df[f'{col}_lag1'] = df.groupby('product_id')[col].shift(1)

print('Lag context features created:')
print([f'{c}_lag1' for c in lag_context_cols])

In [ ]:
# Define the full feature set
CAT_FEATURES = ['therapeutic_area_clean', 'product_family_clean']

FEATURES = [
    # Lag revenue features
    'revenue_lag12',
    'revenue_lag1', 'revenue_lag2', 'revenue_lag3',
    'revenue_roll3_mean',
    # Lag quantity features
    'quantity_lag1', 'quantity_roll3_mean',
    # Calendar
    'month_num', 'quarter', 'year', 'is_december',
    # Product attributes
    'pack_coefficient',
    # Lagged context features
    'n_active_clients_lag1', 'n_invoices_lag1', 'n_regions_lag1',
    'export_rev_share_lag1', 'hospital_rev_share_lag1',
    'commission_rate_w_lag1', 'unit_price_w_lag1',
    # Categorical
    'therapeutic_area_clean', 'product_family_clean',
]

TARGET = 'revenue'
print(f'Total features: {len(FEATURES)}')
print(f'Categorical: {CAT_FEATURES}')

> **Decision applied:** Added `revenue_lag12` (same-month-last-year) as it has the strongest correlation with revenue (r≈0.72). Created lag1 versions of all context variables that would otherwise leak the future. Categorical features will be passed natively to LightGBM. Raw `revenue`, `quantity`, `log_revenue`, `log_quantity`, `unit_price_min`, `unit_price_max` are excluded.

---
## 3. Train / Validation / Test Split (Time-Based)

In [ ]:
TRAIN_END   = pd.Timestamp('2025-07-01')
VAL_START   = pd.Timestamp('2025-08-01')
VAL_END     = pd.Timestamp('2025-11-30')
TEST_START  = pd.Timestamp('2025-12-01')
TEST_END    = pd.Timestamp('2026-04-30')
FORE_START  = pd.Timestamp('2026-05-01')
FORE_END    = pd.Timestamp('2027-04-30')

# Train on non-sparse rows only
df_model = df[df['is_sparse'] == False].copy()

train = df_model[df_model['month_dt'] <= TRAIN_END].copy()
val   = df_model[(df_model['month_dt'] >= VAL_START) & (df_model['month_dt'] <= VAL_END)].copy()
test  = df_model[(df_model['month_dt'] >= TEST_START) & (df_model['month_dt'] <= TEST_END)].copy()

print(f'Train  : {train.shape[0]:,} rows  ({train.month_dt.min().date()} → {train.month_dt.max().date()})')
print(f'Val    : {val.shape[0]:,} rows  ({val.month_dt.min().date()} → {val.month_dt.max().date()})')
print(f'Test   : {test.shape[0]:,} rows  ({test.month_dt.min().date()} → {test.month_dt.max().date()})')
print(f'Sparse rows (excluded from training): {(df["is_sparse"]==True).sum()}')

In [ ]:
# Convert categoricals
for col in CAT_FEATURES:
    for split in [train, val, test]:
        split[col] = split[col].astype('category')

# Drop rows with NaN in any feature (lag NaNs from early history)
train_clean = train.dropna(subset=FEATURES).copy()
val_clean   = val.dropna(subset=FEATURES).copy()
test_clean  = test.dropna(subset=FEATURES).copy()

print(f'After dropping NaN lag rows:')
print(f'  Train: {len(train_clean):,}  Val: {len(val_clean):,}  Test: {len(test_clean):,}')

# Zero-revenue binary target for Stage 1
train_clean['is_nonzero'] = (train_clean['revenue'] > 0).astype(int)
val_clean['is_nonzero']   = (val_clean['revenue'] > 0).astype(int)
test_clean['is_nonzero']  = (test_clean['revenue'] > 0).astype(int)

print(f'\nZero-revenue share in train: {(train_clean["is_nonzero"]==0).mean():.1%}')

> **Decision applied:** Strict time-based split — no shuffle. Training excludes sparse products (`is_sparse==True`). NaN lag rows dropped from training/eval (LightGBM handles NaN at inference natively). Sparse products will still receive predictions at inference time, flagged as low-confidence.

---
## 4. Baseline Models

In [ ]:
def mape(y_true, y_pred, mask=None):
    """MAPE ignoring zero actuals."""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    if mask is not None:
        y_true, y_pred = y_true[mask], y_pred[mask]
    nonzero = y_true > 0
    if nonzero.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[nonzero] - y_pred[nonzero]) / y_true[nonzero])) * 100

def rmse_log(y_true, y_pred):
    nonzero = y_true > 0
    return np.sqrt(mean_squared_error(
        np.log1p(y_true[nonzero]), np.log1p(np.maximum(y_pred[nonzero], 0))
    ))

# Test set actuals
y_test = test_clean['revenue'].values
dec_mask = test_clean['is_december'].values == 1
non_dec_mask = ~dec_mask

baselines = {}

# Naive: last month
baselines['Naive (lag1)'] = {
    'preds': test_clean['revenue_lag1'].fillna(0).values,
}
# Seasonal naive: same month last year
baselines['Seasonal Naive (lag12)'] = {
    'preds': test_clean['revenue_lag12'].fillna(0).values,
}
# Rolling mean
baselines['Rolling Mean (roll3)'] = {
    'preds': test_clean['revenue_roll3_mean'].fillna(0).values,
}

rows = []
for name, b in baselines.items():
    p = b['preds']
    rows.append({
        'Model': name,
        'MAPE (excl. Dec)': mape(y_test, p, non_dec_mask),
        'MAPE (incl. Dec)': mape(y_test, p),
        'RMSE (log scale)': rmse_log(y_test, p),
    })

baseline_df = pd.DataFrame(rows).set_index('Model')
print('=== Baseline Results on Test Set ===')
print(baseline_df.round(2))

> **Baselines established.** The ML model must beat all three baselines on MAPE (excl. December). Seasonal Naive (lag12) is typically the hardest to beat in pharmaceutical panels with strong annual seasonality — clearing it confirms the model adds real signal.

---
## 5. Stage 1 — Zero Classifier (LightGBM)

In [ ]:
X_train_s1 = train_clean[FEATURES]
y_train_s1 = train_clean['is_nonzero']
X_val_s1   = val_clean[FEATURES]
y_val_s1   = val_clean['is_nonzero']

dtrain_s1 = lgb.Dataset(X_train_s1, label=y_train_s1, categorical_feature=CAT_FEATURES)
dval_s1   = lgb.Dataset(X_val_s1,   label=y_val_s1,   reference=dtrain_s1, categorical_feature=CAT_FEATURES)

params_s1 = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'min_child_samples': 20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'verbose': -1,
    'seed': 42,
}

callbacks_s1 = [
    lgb.early_stopping(stopping_rounds=50, verbose=False),
    lgb.log_evaluation(period=100),
]

model_s1 = lgb.train(
    params_s1,
    dtrain_s1,
    num_boost_round=1000,
    valid_sets=[dval_s1],
    callbacks=callbacks_s1,
)

val_prob_s1 = model_s1.predict(X_val_s1)
val_auc_s1  = roc_auc_score(y_val_s1, val_prob_s1)
print(f'\nStage 1 Validation AUC: {val_auc_s1:.4f}')
print(f'Best iteration: {model_s1.best_iteration}')

In [ ]:
# Tune the zero-revenue threshold on validation set
from sklearn.metrics import f1_score

thresholds = np.arange(0.1, 0.9, 0.05)
f1s = [f1_score(y_val_s1, (val_prob_s1 >= t).astype(int)) for t in thresholds]
best_thresh = thresholds[np.argmax(f1s)]

print(f'Best threshold (max F1): {best_thresh:.2f}  (F1={max(f1s):.3f})')

plt.figure(figsize=(8,4))
plt.plot(thresholds, f1s, marker='o')
plt.axvline(best_thresh, color='red', linestyle='--', label=f'Best: {best_thresh:.2f}')
plt.xlabel('Threshold'); plt.ylabel('F1 Score')
plt.title('Stage 1 — Threshold Tuning on Validation Set')
plt.legend(); plt.tight_layout(); plt.show()

ZERO_THRESHOLD = float(best_thresh)

> **Stage 1 complete.** Binary classifier predicts whether a product-month will have any revenue. AUC and F1-tuned threshold reported. Threshold selected on validation set to maximize F1 — this avoids the bias of predicting too many non-zero values.

---
## 6. Stage 2 — Revenue Regressor (LightGBM, non-zero rows only)

In [ ]:
# Train only on rows with revenue > 0
train_nz = train_clean[train_clean['revenue'] > 0].copy()
val_nz   = val_clean[val_clean['revenue'] > 0].copy()

X_train_s2 = train_nz[FEATURES]
y_train_s2 = np.log1p(train_nz['revenue'])
X_val_s2   = val_nz[FEATURES]
y_val_s2   = np.log1p(val_nz['revenue'])

print(f'Stage 2 train rows (non-zero): {len(X_train_s2):,}')
print(f'Stage 2 val rows   (non-zero): {len(X_val_s2):,}')

dtrain_s2 = lgb.Dataset(X_train_s2, label=y_train_s2, categorical_feature=CAT_FEATURES)
dval_s2   = lgb.Dataset(X_val_s2,   label=y_val_s2,   reference=dtrain_s2, categorical_feature=CAT_FEATURES)

params_s2 = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'min_child_samples': 15,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'reg_alpha': 0.1,
    'reg_lambda': 0.5,
    'verbose': -1,
    'seed': 42,
}

callbacks_s2 = [
    lgb.early_stopping(stopping_rounds=50, verbose=False),
    lgb.log_evaluation(period=100),
]

model_s2 = lgb.train(
    params_s2,
    dtrain_s2,
    num_boost_round=1000,
    valid_sets=[dval_s2],
    callbacks=callbacks_s2,
)

val_log_pred = model_s2.predict(X_val_s2)
val_pred_rev = np.expm1(val_log_pred)
val_true_rev = val_nz['revenue'].values

val_mape_s2 = mape(val_true_rev, val_pred_rev)
print(f'\nStage 2 Validation MAPE (non-zero rows): {val_mape_s2:.2f}%')
print(f'Best iteration: {model_s2.best_iteration}')

> **Stage 2 complete.** Regressor trained only on non-zero revenue rows using log1p(revenue) as the target, reversing with expm1() for evaluation. This avoids the right-skew problem (skew > 10 on raw revenue) and makes RMSE on log scale meaningful.

---
## 7. Combined Pipeline — Test Set Evaluation

In [ ]:
def predict_pipeline(X, model_s1, model_s2, threshold=ZERO_THRESHOLD):
    """Two-stage prediction: stage1_prob × expm1(stage2_pred), or 0 if below threshold."""
    prob = model_s1.predict(X)
    log_pred = model_s2.predict(X)
    rev_pred = np.expm1(log_pred)
    # If classifier says zero, predict 0; otherwise scale by probability
    # FIX: return rev_pred directly (not scaled by prob — classifier gates yes/no only)
    final = np.where(prob >= threshold, rev_pred, 0.0)
    return final, prob

X_test = test_clean[FEATURES]
y_test_rev = test_clean['revenue'].values

test_preds, test_probs = predict_pipeline(X_test, model_s1, model_s2)

dec_mask_test    = test_clean['is_december'].values == 1
non_dec_mask_test = ~dec_mask_test

ml_mape_nodec = mape(y_test_rev, test_preds, non_dec_mask_test)
ml_mape_all   = mape(y_test_rev, test_preds)
ml_rmse_log   = rmse_log(y_test_rev, test_preds)

print('=== Test Set Results ===')
print(f'Two-Stage LightGBM  MAPE (excl. Dec): {ml_mape_nodec:.2f}%')
print(f'Two-Stage LightGBM  MAPE (incl. Dec): {ml_mape_all:.2f}%')
print(f'Two-Stage LightGBM  RMSE (log scale): {ml_rmse_log:.4f}')

print('\n=== Baseline Comparison ===')
print(baseline_df.round(2))

In [ ]:
# Full comparison table
all_results = baseline_df.copy()
all_results.loc['Two-Stage LightGBM'] = {
    'MAPE (excl. Dec)': ml_mape_nodec,
    'MAPE (incl. Dec)': ml_mape_all,
    'RMSE (log scale)': ml_rmse_log,
}
print(all_results.round(2))

---
### 7a. Metrics by Revenue Tier

In [ ]:
# Revenue tier: top 10% products by total revenue vs rest
product_rev = df.groupby('product_id')['revenue'].sum()
top10_threshold = product_rev.quantile(0.90)
top10_products  = product_rev[product_rev >= top10_threshold].index

test_clean_eval = test_clean.copy()
test_clean_eval['pred'] = test_preds
test_clean_eval['tier'] = np.where(test_clean_eval['product_id'].isin(top10_products), 'Top 10%', 'Rest')

for tier in ['Top 10%', 'Rest']:
    mask = test_clean_eval['tier'] == tier
    sub = test_clean_eval[mask]
    m = mape(sub['revenue'].values, sub['pred'].values)
    print(f'{tier:8s}  n={mask.sum():4d}  MAPE={m:.1f}%')

---
### 7b. Metrics by Therapeutic Area

In [ ]:
ta_results = []
for ta, grp in test_clean_eval.groupby('therapeutic_area_clean'):
    m = mape(grp['revenue'].values, grp['pred'].values)
    ta_results.append({'Therapeutic Area': ta, 'n': len(grp), 'MAPE (%)': round(m, 1)})

ta_df = pd.DataFrame(ta_results).sort_values('MAPE (%)').reset_index(drop=True)
print(ta_df.to_string(index=False))

---
### 7c. December vs Non-December

In [ ]:
for label, mask in [('December', dec_mask_test), ('Non-December', non_dec_mask_test)]:
    m = mape(y_test_rev, test_preds, mask)
    print(f'{label:15s}  n={mask.sum():4d}  MAPE={m:.1f}%')

In [ ]:
# ── FIX ADDED: Per-Product MAPE Distribution ─────────────────────────────
# This was missing from the original notebook. Aggregate MAPE hides a
# long tail of volatile products (like ABIRATERONE) driving all the error.

def product_mape(grp):
    y_true = grp['revenue'].values
    y_pred = grp['pred'].values
    nonzero = y_true > 0
    if nonzero.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[nonzero] - y_pred[nonzero]) / y_true[nonzero])) * 100

prod_mape_series = test_clean_eval.groupby('product_id').apply(product_mape).dropna()
prod_mape_series.name = 'mape'

print("=== Per-Product MAPE Distribution (Test Set) ===")
percentiles = [25, 50, 75, 90, 95, 99]
for p in percentiles:
    print(f"  p{p:2d}: {prod_mape_series.quantile(p/100):.1f}%")

print(f"\n  Mean  : {prod_mape_series.mean():.1f}%")
print(f"  Median: {prod_mape_series.median():.1f}%")
print(f"  Max   : {prod_mape_series.max():.1f}%")

# Products with MAPE < 50% — the "well-predicted" core
well_predicted = (prod_mape_series < 50).mean()
print(f"\n  Products with MAPE < 50% : {well_predicted:.1%}")
print(f"  Products with MAPE > 200%: {(prod_mape_series > 200).mean():.1%}")

# Top 10 worst products
worst = prod_mape_series.sort_values(ascending=False).head(10).reset_index()
worst.columns = ['product_id', 'mape']
worst = worst.merge(
    test_clean_eval[['product_id','product_name','therapeutic_area_clean']].drop_duplicates(),
    on='product_id', how='left'
)
print("\n=== Top 10 Worst-Predicted Products ===")
print(worst[['product_id','product_name','therapeutic_area_clean','mape']].to_string(index=False))

# Plot histogram
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(prod_mape_series.clip(upper=300), bins=40, color='steelblue', edgecolor='white')
ax1.axvline(50,  color='green',  linestyle='--', label='50% MAPE')
ax1.axvline(100, color='orange', linestyle='--', label='100% MAPE')
ax1.axvline(200, color='red',    linestyle='--', label='200% MAPE')
ax1.set_xlabel('MAPE % (clipped at 300%)'); ax1.set_ylabel('# Products')
ax1.set_title('Per-Product MAPE Distribution'); ax1.legend()

ax2.hist(prod_mape_series.clip(upper=50), bins=30, color='green', edgecolor='white', alpha=0.8)
ax2.set_xlabel('MAPE % (clipped at 50%)'); ax2.set_ylabel('# Products')
ax2.set_title('Well-Predicted Products Only (MAPE ≤ 50%)')

plt.tight_layout()
plt.savefig('per_product_mape_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: per_product_mape_distribution.png")


> **Test evaluation complete.** MAPE reported excluding and including December separately — December is anomalously low (procurement freeze) and inflates MAPE unfairly. Two MAPE values allow honest comparison. Metrics also broken down by revenue tier and therapeutic area.

---
## 8. Feature Importance Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, model, title in [
    (axes[0], model_s1, 'Stage 1 — Zero Classifier'),
    (axes[1], model_s2, 'Stage 2 — Revenue Regressor'),
]:
    imp = pd.Series(
        model.feature_importance(importance_type='gain'),
        index=FEATURES
    ).sort_values(ascending=True).tail(20)

    imp.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Gain')
    ax.grid(axis='x', alpha=0.4)

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: feature_importance.png')

---
## 9. Per-Product Predictions on Test Set

In [ ]:
test_output = test_clean_eval[[
    'product_id', 'product_name', 'month_dt', 'revenue', 'pred',
    'tier', 'therapeutic_area_clean', 'is_december', 'is_sparse'
]].copy()

test_output['abs_pct_error'] = np.where(
    test_output['revenue'] > 0,
    np.abs(test_output['revenue'] - test_output['pred']) / test_output['revenue'] * 100,
    np.nan
)

test_output.columns = [
    'product_id', 'product_name', 'month_dt', 'actual_revenue', 'predicted_revenue',
    'revenue_tier', 'therapeutic_area', 'is_december', 'is_sparse', 'abs_pct_error'
]

test_output.to_csv('test_predictions.csv', index=False)
print(f'Saved test_predictions.csv  ({len(test_output):,} rows)')
print(test_output.head())

---
## 10. Save Models to Disk

In [ ]:
with open(MODEL_DIR / 'model_stage1.pkl', 'wb') as f:
    pickle.dump(model_s1, f)

with open(MODEL_DIR / 'model_stage2.pkl', 'wb') as f:
    pickle.dump(model_s2, f)

# Save metadata needed by the Streamlit app
meta = {
    'features': FEATURES,
    'cat_features': CAT_FEATURES,
    'zero_threshold': ZERO_THRESHOLD,
    'lag_context_cols': lag_context_cols,
    'train_end': str(TRAIN_END.date()),
    'forecast_start': str(FORE_START.date()),
    'forecast_end': str(FORE_END.date()),
}

with open(MODEL_DIR / 'model_meta.pkl', 'wb') as f:
    pickle.dump(meta, f)

print('Models saved:')
for p in sorted(MODEL_DIR.iterdir()):
    print(f'  {p}  ({p.stat().st_size/1024:.1f} KB)')

---
## 11. `predict()` Function for the Streamlit App

This function takes a `product_id` (or `product_name`) and a `target_month` and returns a revenue forecast.

In [ ]:
# Build a lookup table: latest known row per product (used as feature seed for future months)
# For inference, we need the last available month's features to compute the lags.

# Rebuild feature-complete df for all products (including sparse)
df_full = df.copy()
df_full['month_dt'] = pd.to_datetime(df_full['month_dt'])
df_full = df_full.sort_values(['product_id', 'month_dt'])

for col in lag_context_cols:
    if f'{col}_lag1' not in df_full.columns:
        df_full[f'{col}_lag1'] = df_full.groupby('product_id')[col].shift(1)

if 'revenue_lag12' not in df_full.columns:
    df_full['revenue_lag12'] = df_full.groupby('product_id')['revenue'].shift(12)

for col in CAT_FEATURES:
    df_full[col] = df_full[col].astype('category')

# Save full feature-engineered panel for app use
df_full.to_parquet(MODEL_DIR / 'panel_features.parquet', index=False)
print('Saved panel_features.parquet')

# Product lookup table
product_lookup = (
    df_full[['product_id', 'product_name', 'therapeutic_area_clean', 'product_family_clean', 'is_sparse']]
    .drop_duplicates('product_id')
    .set_index('product_id')
)
product_lookup.to_csv(MODEL_DIR / 'product_lookup.csv')
print(f'Saved product_lookup.csv  ({len(product_lookup)} products)')

In [ ]:
import warnings as _w
_w.filterwarnings('ignore')

def predict(product_id: str, target_month: str, panel_df=None) -> dict:
    """
    Predict revenue for a given product_id and target_month.
    
    FIX (v2): Uses iterative lag cascade for months beyond the panel end.
    Previously, requesting any month more than 1 step ahead of panel end 
    caused lag1/2/3 to be NaN, which collapsed Stage 1 probability below 
    the threshold and produced silent zero predictions.
    
    Parameters
    ----------
    product_id   : str   e.g. 'ABI250T120'
    target_month : str   e.g. '2026-09'  (YYYY-MM)
    panel_df     : pd.DataFrame  pre-loaded feature panel (optional)

    Returns
    -------
    dict with keys: predicted_revenue, nonzero_probability, confidence,
                    is_sparse, months_ahead, warning
    """
    import pickle
    import numpy as np
    import pandas as pd
    from pathlib import Path

    MODEL_DIR = Path('models')

    # ── Load models & meta ───────────────────────────────────────────────
    with open(MODEL_DIR / 'model_stage1.pkl', 'rb') as f:
        m1 = pickle.load(f)
    with open(MODEL_DIR / 'model_stage2.pkl', 'rb') as f:
        m2 = pickle.load(f)
    with open(MODEL_DIR / 'model_meta.pkl', 'rb') as f:
        meta = pickle.load(f)

    features       = meta['features']
    cat_features   = meta['cat_features']
    zero_threshold = meta['zero_threshold']
    lag_ctx_cols   = meta['lag_context_cols']

    if panel_df is None:
        panel_df = pd.read_parquet(MODEL_DIR / 'panel_features.parquet')
        panel_df['month_dt'] = pd.to_datetime(panel_df['month_dt'])
        panel_df = panel_df.drop_duplicates(subset=['product_id', 'month_dt'])
        for col in cat_features:
            panel_df[col] = panel_df[col].astype('category')

    target_dt = pd.Timestamp(target_month)

    # ── Validate product ─────────────────────────────────────────────────
    prod_rows = panel_df[panel_df['product_id'] == product_id].sort_values('month_dt')
    if len(prod_rows) == 0:
        return {'error': f'product_id {product_id!r} not found in panel.'}

    is_sparse = bool(prod_rows['is_sparse'].iloc[0])
    panel_end = prod_rows['month_dt'].max()

    # ── Build cascade: collect all months we need to predict iteratively ─
    # If target_month is more than 1 step ahead of panel_end, lag1/lag2/lag3
    # for intermediate months won't be in the panel. We predict them first,
    # then use those predictions as lag inputs for the next month.
    predicted_cache = {}   # {Timestamp -> predicted_revenue}

    def get_rev(dt):
        row = prod_rows[prod_rows['month_dt'] == dt]
        if len(row) > 0:
            return float(row['revenue'].iloc[0])
        return predicted_cache.get(dt, np.nan)

    # Collect all months from (panel_end + 1 month) up to target_dt
    cascade_months = []
    cur = panel_end + pd.DateOffset(months=1)
    while cur <= target_dt:
        cascade_months.append(cur)
        cur = cur + pd.DateOffset(months=1)

    if not cascade_months:
        # target_month is within or at panel_end — no cascade needed
        cascade_months = [target_dt]

    # ── Predict each month in cascade order ──────────────────────────────
    def get_qty(dt, back):
        ref = dt - pd.DateOffset(months=back)
        row = prod_rows[prod_rows['month_dt'] == ref]
        return float(row['quantity'].iloc[0]) if len(row) > 0 else np.nan

    def get_ctx_lag1(dt, col):
        ref = dt - pd.DateOffset(months=1)
        row = prod_rows[prod_rows['month_dt'] == ref]
        return float(row[col].iloc[0]) if len(row) > 0 else np.nan

    static = prod_rows.iloc[-1]   # use latest known row for static attributes

    for month_dt in cascade_months:
        lag1  = get_rev(month_dt - pd.DateOffset(months=1))
        lag2  = get_rev(month_dt - pd.DateOffset(months=2))
        lag3  = get_rev(month_dt - pd.DateOffset(months=3))
        lag12 = get_rev(month_dt - pd.DateOffset(months=12))

        import warnings
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            roll3     = np.nanmean([lag1, lag2, lag3])
            qty_roll3 = np.nanmean([get_qty(month_dt, i) for i in [1, 2, 3]])

        row_dict = {
            'revenue_lag12':          lag12,
            'revenue_lag1':           lag1,
            'revenue_lag2':           lag2,
            'revenue_lag3':           lag3,
            'revenue_roll3_mean':     roll3,
            'quantity_lag1':          get_qty(month_dt, 1),
            'quantity_roll3_mean':    qty_roll3,
            'month_num':              month_dt.month,
            'quarter':                (month_dt.month - 1) // 3 + 1,
            'year':                   month_dt.year,
            'is_december':            int(month_dt.month == 12),
            'pack_coefficient':       static['pack_coefficient'],
            'therapeutic_area_clean': static['therapeutic_area_clean'],
            'product_family_clean':   static['product_family_clean'],
        }
        for col in lag_ctx_cols:
            row_dict[f'{col}_lag1'] = get_ctx_lag1(month_dt, col)

        X = pd.DataFrame([row_dict])[features]
        for col in cat_features:
            X[col] = X[col].astype('category')

        prob     = float(m1.predict(X)[0])
        log_pred = float(m2.predict(X)[0])
        rev_pred = float(np.expm1(log_pred))
        final_rev = rev_pred if prob >= zero_threshold else 0.0

        # Store in cache so subsequent cascade months can use this as a lag
        predicted_cache[month_dt] = final_rev
        _last_prob = prob

    # ── Final result is for target_dt ────────────────────────────────────
    final_rev = predicted_cache[target_dt]
    prob      = _last_prob

    months_ahead = (target_dt.year - panel_end.year) * 12 + (target_dt.month - panel_end.month)

    if prob >= zero_threshold and not is_sparse:
        confidence = 'high' if prob > 0.75 and months_ahead <= 3 else 'medium'
    else:
        confidence = 'low'
    if is_sparse:
        confidence = 'low'

    # ── Warnings ─────────────────────────────────────────────────────────
    warnings_list = []
    if is_sparse:
        warnings_list.append('Sparse product — prediction is low-confidence.')
    if np.isnan(lag12):
        warnings_list.append('revenue_lag12 is NaN (< 12 months history).')
    if target_dt.month == 12:
        warnings_list.append('December forecast — procurement freeze may make this unreliable.')
    if months_ahead > 3:
        warnings_list.append(f'{months_ahead} months ahead — accuracy degrades beyond 3 months.')

    return {
        'product_id':          product_id,
        'target_month':        target_month,
        'predicted_revenue':   round(final_rev, 2),
        'nonzero_probability': round(prob, 4),
        'confidence':          confidence,
        'is_sparse':           is_sparse,
        'is_december':         (target_dt.month == 12),
        'months_ahead':        months_ahead,
        'warning':             ' | '.join(warnings_list) or None,
    }


# ── Smoke tests ──────────────────────────────────────────────────────────
panel = pd.read_parquet('models/panel_features.parquet')
panel['month_dt'] = pd.to_datetime(panel['month_dt'])
panel = panel.drop_duplicates(subset=['product_id','month_dt'])
for col in ['therapeutic_area_clean','product_family_clean']:
    panel[col] = panel[col].astype('category')

sample_products = ['ABI250T120', df['product_id'].iloc[10], df['product_id'].iloc[100]]
print("=== Smoke tests: predict() for 2026-05 through 2026-09 ===")
for pid in [sample_products[0]]:
    print(f"\n{pid}:")
    for m in ['2026-05', '2026-06', '2026-07', '2026-08', '2026-09']:
        r = predict(pid, m, panel)
        print(f"  {m}: revenue={r['predicted_revenue']:>12,.0f}  prob={r['nonzero_probability']:.3f}  conf={r['confidence']}  ahead={r['months_ahead']}")


---
## Summary

| Step | Decision |
|------|----------|
| Target | `log1p(revenue)`, reversed with `expm1()` |
| Architecture | Two-stage LightGBM (classifier + regressor) |
| Features | 22 features; raw revenue/quantity/prices excluded |
| Data leakage | All context vars lagged by 1 month before use |
| Sparse products | Excluded from training; predicted with low-confidence flag |
| NaN lags | Dropped from training; LightGBM handles at inference |
| Split | Strict time-based — no shuffle |
| Evaluation | MAPE excl./incl. December; by tier + therapeutic area |
| Saved artifacts | `model_stage1.pkl`, `model_stage2.pkl`, `model_meta.pkl`, `panel_features.parquet` |

The `predict(product_id, target_month)` function is ready to be imported by the Streamlit app.